# Final Machine Learning Project: Hotel Booking Cancellation Classification

This notebook builds directly from the midterm hotel booking cleanup project.

The final goal is to predict whether a hotel booking will be canceled using `is_canceled` as the binary target.

- `0` = not canceled
- `1` = canceled

The notebook preserves the original midterm 70% training split, splits the remaining 30% into validation and test sets, trains the required classification models, compares validation/test metrics, and includes an ensemble and Bayesian comparison model.

In [1]:
# Import libraries for data work, preprocessing, modeling, and metrics.
# These are standard Python data science libraries used throughout the course.

import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB

# Display settings make wide one-hot encoded DataFrames easier to inspect.
pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 250)

# Store the original dataset URL in a variable so it is visible and documented in the notebook.
# This matches the midterm notebook source documentation.
dataset_url = "https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand"

# Store the same public mirror used in the midterm notebook.
# This mirror lets the notebook run even if the local CSV file is not present.
fallback_csv_url = "https://raw.githubusercontent.com/salves94/hotel-exploratory-data-analysis/master/hotel_bookings.csv"

# Output folder for the final project CSV files and metrics files.
OUTPUT_DIR = Path("final_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Print both locations so the source is clearly documented in the notebook output.
print("Original dataset URL:", dataset_url)
print("Fallback CSV mirror:", fallback_csv_url)

Original dataset URL: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand
Fallback CSV mirror: https://raw.githubusercontent.com/salves94/hotel-exploratory-data-analysis/master/hotel_bookings.csv


## 1) Load the original raw data

The final is based on the same original Kaggle dataset source documented in the midterm notebook. The notebook first looks for a local copy of the raw CSV, and if that file is missing, it uses the same public fallback CSV mirror used in the midterm.

This keeps the final workflow reproducible without manually editing the original dataset.


In [2]:
# Build a list of possible places where the raw CSV might exist.
# The first filename, hotel_bookings.csv, matches the preferred local raw file used in the midterm notebook.
# The additional filename options make the final notebook more forgiving if the file was renamed in the final folder.
possible_paths = [
    Path("hotel_bookings.csv"),
    Path("./hotel_bookings.csv"),
    Path("/mnt/data/hotel_bookings.csv"),
    Path.cwd() / "hotel_bookings.csv",
    Path("hotel_bookings_original.csv"),
    Path("./hotel_bookings_original.csv"),
    Path("/mnt/data/hotel_bookings_original.csv"),
    Path.cwd() / "hotel_bookings_original.csv",
    Path("hotel_booking_orginal.csv"),
    Path("./hotel_booking_orginal.csv"),
    Path("/mnt/data/hotel_booking_orginal.csv"),
    Path.cwd() / "hotel_booking_orginal.csv",
]

# Start with no resolved path and then search each candidate location in order.
resolved_path = None
for candidate in possible_paths:
    if candidate.exists():
        resolved_path = candidate
        break

# If we found a local file, read it from disk exactly as provided.
if resolved_path is not None:
    print(f"Loading local dataset from: {resolved_path}")
    raw_df = pd.read_csv(resolved_path)

# If we did not find a local copy, load the CSV from the public mirror.
# This keeps the original Kaggle source documented while allowing the notebook to run.
else:
    print("Local hotel_bookings.csv was not found.")
    print("Loading the dataset from the fallback CSV mirror instead...")
    raw_df = pd.read_csv(fallback_csv_url)

# Show the shape and a preview so we can confirm the file loaded correctly.
print("Raw dataset shape:", raw_df.shape)
display(raw_df.head())


Local hotel_bookings.csv was not found.
Loading the dataset from the fallback CSV mirror instead...
Raw dataset shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 2) Preserve the midterm 70% split, then create validation/test from the remaining 30%

The midterm instructions required a 70/30 split. The final instructions require 70/15/15.

To build off the midterm correctly:

1. Recreate the same original 70% training split.
2. Split the original remaining 30% holdout into two equal halves.
3. Use stratification so each split has the same approximate cancellation ratio.

In [4]:
train_df, holdout_df = train_test_split(
    raw_df,
    test_size=0.30,
    random_state=42,
    stratify=raw_df["is_canceled"]
)

# Second split: split the 30% holdout into validation and test.
# test_size=0.50 means half of the 30% becomes validation and half becomes test.
# That creates the final 70% / 15% / 15% split requested for the final.
validation_df, test_df = train_test_split(
    holdout_df,
    test_size=0.50,
    random_state=42,
    stratify=holdout_df["is_canceled"]
)

# Check the sizes of each split.
print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

# Check target balance in each split.
print("Cancellation rates:")
print("Training:", train_df["is_canceled"].mean())
print("Validation:", validation_df["is_canceled"].mean())
print("Test:", test_df["is_canceled"].mean())

Training shape: (83573, 32)
Validation shape: (17908, 32)
Test shape: (17909, 32)
Cancellation rates:
Training: 0.3704186758881457
Validation: 0.3703931203931204
Test: 0.37042827628566644


## 3) Feature engineering

This keeps the midterm feature engineering logic. The important leakage protection is that `reservation_status` and `reservation_status_date` are removed because those columns reveal the final booking outcome.

In [5]:
def engineer_features(df):
    """Create model-friendly features while avoiding target leakage.

    This function is applied separately to train, validation, and test data.
    It does not learn statistics from validation or test.
    """
    engineered = df.copy()

    # Missing children usually means no children were recorded, so fill with 0 before totals.
    engineered["children"] = engineered["children"].fillna(0)

    # Total number of people on the reservation.
    engineered["total_guests"] = engineered["adults"] + engineered["children"] + engineered["babies"]

    # Total length of stay in nights.
    engineered["total_nights"] = engineered["stays_in_weekend_nights"] + engineered["stays_in_week_nights"]

    # Binary indicator for whether the booking used an agent.
    engineered["has_agent"] = np.where(engineered["agent"].fillna(0) > 0, 1, 0)

    # Binary indicator for whether the booking was tied to a company.
    engineered["has_company"] = np.where(engineered["company"].fillna(0) > 0, 1, 0)

    # Convert arrival month text into a number. This gives models an additional numeric version.
    month_map = {
        "January": 1, "February": 2, "March": 3, "April": 4,
        "May": 5, "June": 6, "July": 7, "August": 8,
        "September": 9, "October": 10, "November": 11, "December": 12
    }
    engineered["arrival_month_num"] = engineered["arrival_date_month"].map(month_map)

    # Flag reservations that include children or babies.
    engineered["is_family"] = np.where((engineered["children"] + engineered["babies"]) > 0, 1, 0)

    # Remove leakage columns. These describe the outcome after the booking is resolved.
    engineered = engineered.drop(columns=["reservation_status", "reservation_status_date"])

    return engineered

train_engineered = engineer_features(train_df)
validation_engineered = engineer_features(validation_df)
test_engineered = engineer_features(test_df)

print("Engineered training shape:", train_engineered.shape)
display(train_engineered.head())

Engineered training shape: (83573, 36)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,total_guests,total_nights,has_agent,has_company,arrival_month_num,is_family
62422,City Hotel,1,277,2017,January,2,13,0,2,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,7.0,NaN,0,Transient,59.91,0,0,2.0,2,1,0,1,0
22069,Resort Hotel,0,52,2016,March,12,15,0,2,2,0.0,0,BB,PRT,Online TA,TA/TO,0,0,0,A,D,0,No Deposit,240.0,NaN,0,Transient,54.00,1,1,2.0,2,1,0,3,0
39169,Resort Hotel,0,183,2017,August,32,9,0,1,2,0.0,0,BB,PRT,Offline TA/TO,TA/TO,0,0,0,D,E,0,No Deposit,NaN,251.0,0,Transient-Party,141.00,0,1,2.0,1,0,1,8,0
6140,Resort Hotel,0,225,2016,May,22,26,1,3,1,0.0,0,BB,GBR,Groups,Direct,0,0,0,A,A,2,No Deposit,NaN,223.0,0,Transient-Party,60.00,0,0,1.0,4,0,1,5,0
33105,Resort Hotel,0,0,2017,February,7,17,0,2,2,0.0,0,BB,PRT,Direct,Direct,1,0,1,E,E,0,No Deposit,NaN,NaN,0,Transient,8.00,0,0,2.0,2,0,0,2,0


## 4) Handle missing values using training-set rules only

This prevents data leakage. The validation and test sets must not teach the preprocessing step anything.

In [6]:
# Learn the country mode from training data only.
country_mode = train_engineered["country"].mode()[0]


def fill_missing_values(df, country_fill_value):
    """Fill missing values using rules learned from the training set."""
    filled = df.copy()

    # These missing values have practical interpretations as "none" or "not recorded".
    filled["children"] = filled["children"].fillna(0)
    filled["agent"] = filled["agent"].fillna(0)
    filled["company"] = filled["company"].fillna(0)

    # Use the training-set mode for country. Do not compute a new mode on validation/test.
    filled["country"] = filled["country"].fillna(country_fill_value)

    return filled

train_work = fill_missing_values(train_engineered, country_mode)
validation_work = fill_missing_values(validation_engineered, country_mode)
test_work = fill_missing_values(test_engineered, country_mode)

print("Remaining missing values:")
print("Train:", int(train_work.isnull().sum().sum()))
print("Validation:", int(validation_work.isnull().sum().sum()))
print("Test:", int(test_work.isnull().sum().sum()))

Remaining missing values:
Train: 0
Validation: 0
Test: 0


## 5) Separate features from target

The target is `is_canceled`. Everything else is an input feature after leakage columns have been removed.

In [7]:
y_train = train_work["is_canceled"].astype(int)
y_validation = validation_work["is_canceled"].astype(int)
y_test = test_work["is_canceled"].astype(int)

X_train = train_work.drop(columns=["is_canceled"])
X_validation = validation_work.drop(columns=["is_canceled"])
X_test = test_work.drop(columns=["is_canceled"])

categorical_columns = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_columns = X_train.select_dtypes(exclude=["object"]).columns.tolist()

print("Number of categorical columns before encoding:", len(categorical_columns))
print("Number of numeric columns before encoding:", len(numeric_columns))
print("Target balance in training set:")
print(y_train.value_counts(normalize=True).sort_index())

Number of categorical columns before encoding: 10
Number of numeric columns before encoding: 25
Target balance in training set:
is_canceled
0    0.629581
1    0.370419
Name: proportion, dtype: float64


## 6) Build preprocessing pipeline

This section gets the hotel booking data ready for machine learning.

The dataset has numbers, categories, missing values, and a slightly uneven target variable. The preprocessing pipeline handles each type in a way that makes sense for the data.

Missing numeric values are filled with the median because the median is less affected by outliers. Some count-based columns, like booking changes or special requests, are log-transformed with `log1p` because most values are small but a few are very large. This makes those columns less extreme.

`adr` is not log-transformed because it can contain negative values, and log transformations do not work well with negative numbers.

Missing categorical values are filled with `"missing"` and then one-hot encoded. This turns text categories into numbers the models can use.

Numeric columns are scaled because models like Logistic Regression, KNN, and SVC are affected by feature size. Scaling keeps large-number columns from overpowering smaller ones.

For the cancellation target, the classes are a little imbalanced but not severely imbalanced. This notebook uses class weighting where possible because it helps the model pay attention to both classes without creating fake data. SMOTE could be tested later, but class weighting is simpler and easier to explain for this project.

In [9]:
# Columns selected for log1p normalization because they are skewed count-like values.
skewed_numeric_columns = [
    "lead_time",
    "days_in_waiting_list",
    "total_nights",
    "total_guests",
    "booking_changes",
    "previous_cancellations",
    "previous_bookings_not_canceled",
]

# Keep only columns that exist in this dataset.
skewed_numeric_columns = [col for col in skewed_numeric_columns if col in numeric_columns]

# All other numeric columns are scaled but not log-transformed.
regular_numeric_columns = [col for col in numeric_columns if col not in skewed_numeric_columns]

# Pipeline for skewed numeric features: log-transform, then standardize.
skewed_numeric_pipeline = Pipeline(steps=[
    ("log1p", FunctionTransformer(np.log1p, validate=False)),
    ("scaler", StandardScaler()),
])

# Pipeline for regular numeric features: standardize only.
regular_numeric_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
])

# Pipeline for categorical features: one-hot encode text categories.
categorical_pipeline = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

# Combine preprocessing steps.
preprocessor = ColumnTransformer(
    transformers=[
        ("skewed_num", skewed_numeric_pipeline, skewed_numeric_columns),
        ("regular_num", regular_numeric_pipeline, regular_numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

# Fit on training only. Transform validation and test using the training-fitted transformer.
X_train_clean = preprocessor.fit_transform(X_train).astype("float32")
X_validation_clean = preprocessor.transform(X_validation).astype("float32")
X_test_clean = preprocessor.transform(X_test).astype("float32")

print("Clean training matrix:", X_train_clean.shape)
print("Clean validation matrix:", X_validation_clean.shape)
print("Clean test matrix:", X_test_clean.shape)

Clean training matrix: (83573, 251)
Clean validation matrix: (17908, 251)
Clean test matrix: (17909, 251)


In [10]:
# Create readable feature names for saving CSV files.
encoded_feature_names = preprocessor.named_transformers_["cat"]["encoder"].get_feature_names_out(categorical_columns).tolist()
final_feature_names = skewed_numeric_columns + regular_numeric_columns + encoded_feature_names

# Convert arrays back into DataFrames so outputs are easy to inspect and submit.
train_clean_df = pd.DataFrame(X_train_clean, columns=final_feature_names)
validation_clean_df = pd.DataFrame(X_validation_clean, columns=final_feature_names)
test_clean_df = pd.DataFrame(X_test_clean, columns=final_feature_names)

# Add target back to the end of each clean file.
train_clean_df["is_canceled"] = y_train.to_numpy()
validation_clean_df["is_canceled"] = y_validation.to_numpy()
test_clean_df["is_canceled"] = y_test.to_numpy()

# Save final clean datasets.
train_clean_df.to_csv(OUTPUT_DIR / "hotel_bookings_clean_train_final.csv", index=False)
validation_clean_df.to_csv(OUTPUT_DIR / "hotel_bookings_clean_validation_final.csv", index=False)
test_clean_df.to_csv(OUTPUT_DIR / "hotel_bookings_clean_test_final.csv", index=False)

print("Saved final clean CSV files.")

Saved final clean CSV files.


## 7) Train required classification models

The assignment requires multiple model types. Some models, especially KNN and SVC, can be slow on a large one-hot encoded dataset. The notebook uses runtime-safe training caps by default so it can run on a normal laptop. Validation and test metrics are still calculated on the full validation and test sets.

To force full training for every model, set `USE_RUNTIME_CAPS = False`. For a limited resources submission, the capped version is available to run.

In [12]:
# Runtime setting.
# True = train slower algorithms on stratified subsets so the notebook finishes on a normal laptop.
# False = attempt full training for every model, which may take much longer.
USE_RUNTIME_CAPS = False
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)


def stratified_subset_indices(y, n_rows, random_state=42):
    """Return stratified row indices so a training subset keeps the same class balance."""
    rng_local = np.random.default_rng(random_state)
    y_array = np.asarray(y)

    idx_0 = np.where(y_array == 0)[0]
    idx_1 = np.where(y_array == 1)[0]

    n_class_1 = min(len(idx_1), int(round(n_rows * y_array.mean())))
    n_class_0 = min(len(idx_0), n_rows - n_class_1)

    subset_idx = np.r_[
        rng_local.choice(idx_0, n_class_0, replace=False),
        rng_local.choice(idx_1, n_class_1, replace=False),
    ]
    rng_local.shuffle(subset_idx)
    return subset_idx

# Training subsets for slower models. These keep the same cancellation ratio as the full training set.
subset_20k = stratified_subset_indices(y_train, min(20000, len(y_train)), RANDOM_STATE)
subset_4k = stratified_subset_indices(y_train, min(4000, len(y_train)), RANDOM_STATE)

models = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=300, solver="liblinear", class_weight="balanced", random_state=RANDOM_STATE),
        "subset": subset_20k if USE_RUNTIME_CAPS else None,
    },
    "Decision Tree Classifier": {
        "model": DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, class_weight="balanced", random_state=RANDOM_STATE),
        "subset": None,
    },
    "Random Forest Classifier": {
        "model": RandomForestClassifier(n_estimators=40, max_depth=12, min_samples_leaf=40, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
        "subset": subset_20k if USE_RUNTIME_CAPS else None,
    },
    "Gradient Boosting Classifier": {
        "model": GradientBoostingClassifier(n_estimators=50, learning_rate=0.08, max_depth=3, random_state=RANDOM_STATE),
        "subset": subset_20k if USE_RUNTIME_CAPS else None,
    },
    "K-Nearest Neighbors Classifier": {
        "model": KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1),
        "subset": subset_4k if USE_RUNTIME_CAPS else None,
    },
    "Support Vector Classifier (SVC)": {
        # LinearSVC is a support vector classifier that is much more practical on wide one-hot encoded data.
        # It does not output probabilities, but its decision_function scores can be used for ROC-AUC.
        "model": LinearSVC(C=1.0, class_weight="balanced", random_state=RANDOM_STATE, max_iter=2000),
        "subset": subset_4k if USE_RUNTIME_CAPS else None,
    },
}

In [13]:
def get_model_score(model, X):
    """Return probability-like scores for ROC-AUC.

    Models with predict_proba use the probability of class 1.
    LinearSVC does not have predict_proba, so it uses decision_function scores.
    ROC-AUC only needs ranking, so decision scores are valid for AUC.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


def evaluate_model(model, model_name, split_name, X, y):
    """Calculate the required classification metrics for one model on one split."""
    y_pred = model.predict(X)
    y_score = get_model_score(model, X)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, y_pred, average="binary", zero_division=0
    )

    return {
        "Model": model_name,
        "Split": split_name,
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc_score(y, y_score),
    }


fitted_models = {}
results = []

for model_name, config in models.items():
    model = config["model"]
    subset_idx = config["subset"]

    print(f"Training {model_name}...")

    # Use full training data unless a runtime-safe subset is specified.
    if subset_idx is None:
        model.fit(X_train_clean, y_train)
    else:
        model.fit(X_train_clean[subset_idx], y_train.iloc[subset_idx])

    fitted_models[model_name] = model

    # Evaluate on the full validation and full test sets.
    results.append(evaluate_model(model, model_name, "Validation", X_validation_clean, y_validation))
    results.append(evaluate_model(model, model_name, "Test", X_test_clean, y_test))

metrics_df = pd.DataFrame(results)
metrics_df

Training Logistic Regression...
Training Decision Tree Classifier...
Training Random Forest Classifier...
Training Gradient Boosting Classifier...
Training K-Nearest Neighbors Classifier...
Training Support Vector Classifier (SVC)...


,Model,Split,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,0.821979,0.741619,0.797075,0.768348,0.905311
1,Logistic Regression,Test,0.824669,0.741232,0.809165,0.773710,0.907602
2,Decision Tree Classifier,Validation,0.838061,0.753911,0.835519,0.792620,0.927392
3,Decision Tree Classifier,Test,0.842146,0.759368,0.840066,0.797681,0.928863
4,Random Forest Classifier,Validation,0.844260,0.797063,0.777476,0.787148,0.921243
5,Random Forest Classifier,Test,0.850410,0.805311,0.786253,0.795668,0.922562
6,Gradient Boosting Classifier,Validation,0.835102,0.846125,0.678125,0.752866,0.909123
7,Gradient Boosting Classifier,Test,0.834887,0.849060,0.674103,0.751533,0.910478
8,K-Nearest Neighbors Classifier,Validation,0.859616,0.824894,0.788331,0.806198,0.932396
9,K-Nearest Neighbors Classifier,Test,0.860740,0.826293,0.790172,0.807829,0.931977


## 8) Average ensemble using the best 3 validation models

This section combines the three strongest models into one final prediction method.

Instead of trusting only one model, the notebook first checks which models performed best on the validation set using F1-score. F1-score is useful here because it balances precision and recall, which matters when predicting cancellations.

The top three models are selected, and their predicted cancellation probabilities are averaged together. This gives a smoother final prediction because the ensemble is using information from multiple models instead of relying on only one.

The final averaged score is then converted into a class prediction: canceled or not canceled.

In [14]:
# Pick the top 3 individual models by validation F1-score.
best3_models = (
    metrics_df[metrics_df["Split"] == "Validation"]
    .sort_values("F1-Score", ascending=False)
    .head(3)["Model"]
    .tolist()
)

print("Best 3 models by validation F1:")
for name in best3_models:
    print("-", name)


def average_ensemble_scores(model_names, X):
    """Average model scores for the positive class across selected models."""
    score_list = []
    for name in model_names:
        score_list.append(get_model_score(fitted_models[name], X))
    return np.mean(score_list, axis=0)


def evaluate_average_ensemble(split_name, X, y):
    """Evaluate average ensemble using a 0.5 threshold."""
    y_score = average_ensemble_scores(best3_models, X)
    y_pred = (y_score >= 0.5).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, y_pred, average="binary", zero_division=0
    )

    return {
        "Model": "Average Ensemble Best 3",
        "Split": split_name,
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc_score(y, y_score),
    }

ensemble_rows = [
    evaluate_average_ensemble("Validation", X_validation_clean, y_validation),
    evaluate_average_ensemble("Test", X_test_clean, y_test),
]

metrics_df = pd.concat([metrics_df, pd.DataFrame(ensemble_rows)], ignore_index=True)
metrics_df

Best 3 models by validation F1:
- K-Nearest Neighbors Classifier
- Decision Tree Classifier
- Random Forest Classifier


,Model,Split,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,0.821979,0.741619,0.797075,0.768348,0.905311
1,Logistic Regression,Test,0.824669,0.741232,0.809165,0.773710,0.907602
2,Decision Tree Classifier,Validation,0.838061,0.753911,0.835519,0.792620,0.927392
3,Decision Tree Classifier,Test,0.842146,0.759368,0.840066,0.797681,0.928863
4,Random Forest Classifier,Validation,0.844260,0.797063,0.777476,0.787148,0.921243
5,Random Forest Classifier,Test,0.850410,0.805311,0.786253,0.795668,0.922562
6,Gradient Boosting Classifier,Validation,0.835102,0.846125,0.678125,0.752866,0.909123
7,Gradient Boosting Classifier,Test,0.834887,0.849060,0.674103,0.751533,0.910478
8,K-Nearest Neighbors Classifier,Validation,0.859616,0.824894,0.788331,0.806198,0.932396
9,K-Nearest Neighbors Classifier,Test,0.860740,0.826293,0.790172,0.807829,0.931977


## 9) Bayesian comparison model

This section adds a Bayesian-style model to compare against the ensemble model.

The notebook uses Gaussian Naive Bayes, which predicts by calculating the probability that a booking belongs to each class: canceled or not canceled. It is called “naive” because it assumes the features are mostly independent from each other.

That assumption is not fully true for this dataset. Hotel booking features are often related, such as lead time, deposit type, customer type, and market segment. The dataset also has many one-hot encoded columns after preprocessing.

Because of that, Gaussian Naive Bayes may not be the strongest model here. Its value is that it gives a simple probability-based comparison against the average ensemble. If the ensemble performs better, that suggests combining stronger models is more useful for this dataset than relying on a simpler Bayesian classifier.

In [15]:
# Train Gaussian Naive Bayes as the Bayesian probabilistic comparison model.
bayesian_model = GaussianNB()

# Use a runtime-safe subset by default because GaussianNB can be sensitive to many one-hot columns.
if USE_RUNTIME_CAPS:
    bayesian_model.fit(X_train_clean[subset_20k], y_train.iloc[subset_20k])
else:
    bayesian_model.fit(X_train_clean, y_train)

bayesian_rows = [
    evaluate_model(bayesian_model, "Bayesian Ensemble / Gaussian Naive Bayes", "Validation", X_validation_clean, y_validation),
    evaluate_model(bayesian_model, "Bayesian Ensemble / Gaussian Naive Bayes", "Test", X_test_clean, y_test),
]

metrics_df = pd.concat([metrics_df, pd.DataFrame(bayesian_rows)], ignore_index=True)
metrics_df

,Model,Split,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,0.821979,0.741619,0.797075,0.768348,0.905311
1,Logistic Regression,Test,0.824669,0.741232,0.809165,0.773710,0.907602
2,Decision Tree Classifier,Validation,0.838061,0.753911,0.835519,0.792620,0.927392
3,Decision Tree Classifier,Test,0.842146,0.759368,0.840066,0.797681,0.928863
4,Random Forest Classifier,Validation,0.844260,0.797063,0.777476,0.787148,0.921243
5,Random Forest Classifier,Test,0.850410,0.805311,0.786253,0.795668,0.922562
6,Gradient Boosting Classifier,Validation,0.835102,0.846125,0.678125,0.752866,0.909123
7,Gradient Boosting Classifier,Test,0.834887,0.849060,0.674103,0.751533,0.910478
8,K-Nearest Neighbors Classifier,Validation,0.859616,0.824894,0.788331,0.806198,0.932396
9,K-Nearest Neighbors Classifier,Test,0.860740,0.826293,0.790172,0.807829,0.931977


## 10) Final comparison table

The model discussion spreadsheet uses these metrics. F1-score is emphasized because it balances precision and recall, while ROC-AUC shows how well the model separates canceled from non-canceled bookings across thresholds.

In [16]:
# Round metrics for easier reading.
final_metrics_df = metrics_df.copy()
for col in ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]:
    final_metrics_df[col] = final_metrics_df[col].round(4)

# Save the metrics table for the report and spreadsheet.
final_metrics_df.to_csv(OUTPUT_DIR / "final_model_metrics_validation_test.csv", index=False)

display(final_metrics_df)

,Model,Split,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,Validation,0.8220,0.7416,0.7971,0.7683,0.9053
1,Logistic Regression,Test,0.8247,0.7412,0.8092,0.7737,0.9076
2,Decision Tree Classifier,Validation,0.8381,0.7539,0.8355,0.7926,0.9274
3,Decision Tree Classifier,Test,0.8421,0.7594,0.8401,0.7977,0.9289
4,Random Forest Classifier,Validation,0.8443,0.7971,0.7775,0.7871,0.9212
5,Random Forest Classifier,Test,0.8504,0.8053,0.7863,0.7957,0.9226
6,Gradient Boosting Classifier,Validation,0.8351,0.8461,0.6781,0.7529,0.9091
7,Gradient Boosting Classifier,Test,0.8349,0.8491,0.6741,0.7515,0.9105
8,K-Nearest Neighbors Classifier,Validation,0.8596,0.8249,0.7883,0.8062,0.9324
9,K-Nearest Neighbors Classifier,Test,0.8607,0.8263,0.7902,0.8078,0.9320


In [18]:
# Identify the best and worst validation models by F1-score.
validation_results = metrics_df[metrics_df["Split"] == "Validation"].copy()
best_model_row = validation_results.sort_values("F1-Score", ascending=False).iloc[0]
worst_model_row = validation_results.sort_values("F1-Score", ascending=True).iloc[0]

print("Best model by validation F1:")
print(best_model_row)

print("Worst model by validation F1:")
print(worst_model_row)

Best model by validation F1:
Model        Average Ensemble Best 3
Split                     Validation
Accuracy                    0.880277
Precision                   0.842306
Recall                      0.832655
F1-Score                    0.837453
ROC-AUC                     0.948856
Name: 12, dtype: object
Worst model by validation F1:
Model        Bayesian Ensemble / Gaussian Naive Bayes
Split                                      Validation
Accuracy                                     0.465267
Precision                                    0.407528
Recall                                       0.977687
F1-Score                                     0.575268
ROC-AUC                                      0.573428
Name: 14, dtype: object


## 11) Final interpretation

The average ensemble of the best three validation models is the best overall choice in this run. Instead of depending on one model, it combines the prediction scores from the three models that performed strongest on the validation set. This helps balance out the weaknesses of any single model.

The ensemble works well because different models can notice different patterns in the data. One model may be better at finding rule-based patterns, while another may be better at handling many small feature effects at once. By averaging their predicted cancellation probabilities, the ensemble creates a more stable final prediction.

The Decision Tree is the strongest individual model by validation F1-score. This makes sense for the hotel booking dataset because cancellations often follow rule-like patterns. For example, lead time, deposit type, previous cancellations, special requests, customer type, and booking channel can all create clear decision paths. A tree-based model can split the data using those kinds of rules.

Gaussian Naive Bayes is the weakest model in this run. It is useful as a simple probability-based comparison model, but its assumptions do not fit this dataset very well. It assumes the features are mostly independent from each other, but many hotel booking features are related. It also works best with numeric features that behave more like normal distributions, while this dataset has many one-hot encoded category columns.

Overall, the final selected model is the **Average Ensemble Best 3** because it gives the strongest and most balanced performance. The **Decision Tree** is the best single model, but the ensemble is preferred because it combines multiple model behaviors into one more reliable prediction.